# Qwen2.5-1.5B-Instruct dans NeuroDSL

Charge le vrai checkpoint HuggingFace `Qwen/Qwen2.5-1.5B-Instruct` (chargé une fois, converti au format natif NeuroDSL) dans un `NeuroGraph`, et propose une fonction `chat(prompt)` pour lui parler directement depuis ce notebook.

**Ce qui est déjà vérifié avant ce notebook** (voir `notebook/qwen2_parity_check.jl` et le rapport de session) : les logits de ce graphe concordent avec le modèle HuggingFace de référence à la précision flottante (écart absolu max mesuré : `3.7e-5`), le top-1/top-5 et une continuation gloutonne de 8 tokens sont identiques token pour token sur plusieurs prompts.

**Portée livrée : multi-tours.** L'historique de conversation (`HISTORY`) est réellement réutilisé d'un appel à `chat(...)` au suivant (gabarit ChatML réel). `reset_chat!()` vide l'historique.

**Deux correctifs de performance appliqués et vérifiés ce soir, effet COMBINÉ mesuré directement dans ce notebook (cellules §5) :**
1. **Préfixe en un seul passage batché** (`NeuroDSL.prime_kv_cache_from_prefix!`) au lieu d'un remplissage du cache token par token (bug initial : ~19-28s pour un préfixe de 36 tokens). Vérifié correct : `notebook/kv_cache_prefix_prime_qwen_gate.jl`, écart max `5.6e-5` vs remplissage séquentiel.
2. **Serveur tokenizer persistant** (`qwen2_tokenizer_helper.py --serve`) au lieu de relancer Python + réimporter `transformers` + recharger le tokenizer à CHAQUE appel (~13-14s de coût fixe par appel, indépendant du cache KV). Vérifié correct : `notebook/kv_cache_tokenizer_persistent_check_run.log`, résultats identiques au mode un-coup sur encode_chat/decode/eos_id.

**Résultat combiné, mesuré dans CE notebook (§5, même prompt "What is the capital of Egypt?", 3 appels dans la même session) :**

| Appel | Temps total | Contexte |
|---|---|---|
| Cellule 8 (1er appel de la session) | **14.2s** | JIT CUDA froid + tokenizer froid (les deux coûts uniques payés une fois) |
| Appel A (§5, même session, plus tard) | **4.2s** | tout déjà chaud |
| Appel B (§5, répétition) | **4.1s** | stable |

Soit **~3.4x plus rapide** dès le deuxième message, et stable ensuite -- gain réel, visible sur le temps total perçu par l'utilisateur, pas seulement dans une mesure de calcul isolée. Le premier message d'une session reste plus lent (compilation JIT + démarrage du tokenizer, tous deux payés une seule fois) ; c'est attendu et documenté, pas un bug.

**Portée précise du cache (pour ne pas surclaimer) :** le cache KV est reconstruit à neuf à CHAQUE appel à `chat(...)` à partir de l'historique ChatML complet -- il ne persiste PAS encore D'UN appel à L'AUTRE (économiserait le retraitement de l'historique des tours précédents ; pas construit, pas validé). Le serveur tokenizer, lui, persiste bien d'un appel à l'autre (c'est le correctif ci-dessus).

**CORRECTIF MÉMOIRE 2026-08-31 -- double allocation des poids éliminée.** Investigation partie d'une observation utilisateur (VRAM ~13-14 Go pour ce chat, alors que le budget "poids seuls" attendu en Float32 pour 1.5B paramètres est ~5.75-6.78 Go). Cause trouvée par mesure directe (`notebook/diag_kv_cache_double_alloc_phaseA.jl`/`_phaseB.jl`) : ce notebook gardait DEUX namespaces séparés (`ns_load`/`ns_chat`) avec les poids COPIÉS PAR VALEUR de l'un à l'autre (`copy_params_to_namespace!`) -- une dette technique du correctif du 2026-07-29 (voir ligne "historique des bugs" ci-dessous), qui n'était plus nécessaire depuis qu'un correctif ultérieur de `demand!` (restriction au cône des ancêtres réels, `_ancestors_of!`) a rendu le partage d'un namespace unique sûr, mais que personne n'était revenu retirer. Résultat mesuré, poids réels Qwen2.5-1.5B, même prompt, deux process Julia isolés : **pic VRAM 14420 Mo -> 10324 Mo (-4.1 Go, -28.4%)**, tokens générés et logits à chaque pas **numériquement identiques** (écart abs max = 0.0) à l'ancien chemin, aucune régression de vitesse (0.475 vs 0.488 s/tok en régime établi). Ce notebook utilise maintenant ce chemin corrigé (cellule §1) -- `ns_load`/`ns_chat` restent définis pour ne pas renommer tout le notebook, mais pointent vers le MÊME namespace.

**CORRECTIF MÉMOIRE 2026-08-31 (bis) -- fuite mémoire multi-tours dans le cache KV (`aux_data[:history]` jamais libéré).** Investigation partie d'une VRAIE conversation à 4 tours de l'utilisateur dans la cellule d'interface interactive ci-dessous (§5, sortie sauvegardée telle quelle : "hi qwen" / "how are you" / "integral x*exp(x)" / "integral x*exp(-x)") : VRAM observée passant de ~8.5 Go à ~13.3 Go précisément autour du 4e tour (la 2e réponse longue, LaTeX, ~200-300 tokens), un saut sans rapport avec la croissance attendue du cache KV (quelques centaines de Ko/token) mesurée précédemment sur un test à UN SEUL tour. Reproduit en process isolé, méthodologie identique (`notebook/diag_multiturn_vram_growth.jl`, mode `multiturn` -- même 4 prompts, `max_new_tokens=200`) : **pic nvidia-smi 16021 Mo** sur le chemin AVANT correctif (rejoue la fuite), contre **8671 Mo** mesuré ce même après-midi sur UN SEUL tour -- confirme le saut multi-tours signalé par l'utilisateur.

Cause racine trouvée par lecture directe du code, pas par supposition : `kv_cache_append!` (`src/kv_cache.jl`) et `prime_kv_cache_from_prefix!` (`src/layers.jl`) réassignent `out_node.aux_data[:history]` (le K/V accumulé du cache, une COPIE distincte de `.value`, jamais suivie par le mécanisme générique de `execute_rule!`) à CHAQUE token généré (`kv_cache_append!`, appelé pour CHACUN des `2 * n_layers * n_kv_heads = 112` nœuds K/V, à CHAQUE pas de génération -- jusqu'à 200 fois par tour) et à CHAQUE tour (`prime_kv_cache_from_prefix!`, une fois par tour), SANS appeler `Backend.free!` sur l'ancienne valeur -- contrairement à `.value` (`execute_rule!`, `src/dispatch.jl:243-258`), qui LUI est explicitement libéré de façon SYNCHRONE avant réallocation, précisément pour ne pas dépendre d'un passage du GC Julia qui "peut ne jamais arriver au milieu d'une boucle chaude" (citation du commentaire de `Backend.free!`, `src/backend.jl`). La cellule d'interface interactive (§5, en bas) n'appelle JAMAIS `GC.gc()`/`CUDA.reclaim()` entre les tours (seule la cellule séparée juste au-dessus le fait, manuellement, pas dans la boucle `chat(...)`) -- l'historique K/V superflu de chaque pas de génération s'accumulait donc en mémoire GPU non réclamée, potentiellement sur toute la durée d'une session interactive.

**Correctif** : `Backend.free!` appelé de façon synchrone sur l'ancien `aux_data[:history]` juste avant réassignation, aux deux sites d'écriture (`src/kv_cache.jl:kv_cache_append!`, `src/layers.jl:prime_kv_cache_from_prefix!`) -- même discipline que `.value`, zéro dépendance au GC. **Mesuré, MÊME scénario à 4 tours, process isolé** (`notebook/diag_multiturn_vram_growth_baseline_run.log` avant vs `notebook/diag_multiturn_vram_growth_fixed_run.log` après) :

| Tour | Avant (nvidia-smi) | Après (nvidia-smi) |
|---|---|---|
| 1 ("hi qwen") | 10363 Mo | 8701 Mo |
| 2 ("how are you") | 10363 Mo | 8701 Mo |
| 3 (intégrale, 200 tok) | 10396 Mo | 8701 Mo |
| 4 (intégrale, 200 tok, préfixe 314 tok) | **16021 Mo** | **9177 Mo** |

Soit **pic VRAM 16021 Mo -> 9177 Mo (-6.84 Go, -42.7%)** sur la conversation EXACTE qui a déclenché l'investigation, sans aucune régression de vitesse (le correctif ne fait que déplacer QUAND un buffer déjà mort est libéré, jamais QUOI est calculé). Réponses générées **identiques mot pour mot avant/après** (même nombre de tokens par tour : 9/27/200/200, mêmes longueurs de préfixe : 32/54/97/314, mêmes points d'arrêt EOS -- attendu : le correctif ne touche que le cycle de vie mémoire, jamais une valeur numérique). Contrôle séparé (`notebook/diag_multiturn_vram_growth_single_long_turn_baseline_run.log`, mode `single-long-turn`) : UN SEUL tour de 250 tokens depuis un préfixe court (40 tokens, PAS d'historique multi-tours accumulé) ne produit qu'une croissance modeste (+278 Mo pool_current brut, avant tout GC) -- élimine l'hypothèse "générer une réponse longue suffit à lui seul" : c'est la COMBINAISON préfixe déjà long (accumulé sur plusieurs tours) + nombreux pas de génération qui fait exploser le volume de `aux_data[:history]` abandonné par pas. Le reliquat de VRAM au-delà du premier tour (8701 Mo en régime établi vs 8671 Mo mesuré sur un seul tour cet après-midi) est un vrai coût, proportionné : le préfixe recalculé grandit réellement à chaque tour (32 -> 54 -> 97 -> 314 tokens), et le cache KV porte réellement tout cet historique -- pas un reliquat de fuite.

**Historique complet des bugs trouvés et corrigés** (pour ne pas les répéter) : cache rempli token par token pour le préfixe (corrigé -> passage batché) ; partage de namespace entre le graphe caché et le graphe de recalcul complet, qui faisait ré-exécuter incidemment des nœuds à état lors d'un `demand!` sans rapport (corrigé À L'ÉPOQUE, 2026-07-29, en séparant les namespaces + `copy_params_to_namespace!` -- ce correctif doublait le budget mémoire des poids, voir le correctif mémoire 2026-08-31 ci-dessus qui l'élimine maintenant que sa cause racine dans `demand!` est réglée) ; tokenizer relancé à froid à chaque appel (corrigé -> serveur persistant) ; `aux_data[:history]` du cache KV jamais libéré synchroniquement, accumulation non bornée sur une conversation multi-tours (corrigé -> `Backend.free!` synchrone, voir correctif 2026-08-31 bis ci-dessus). Tous ont été trouvés en mesurant, pas en supposant.

**Tokenizer : aucune nouvelle dépendance Julia**, dans les deux modes (`qwen2_tokenizer_helper.py`, environnement conda isolé `neurodsl_llm_check` -- `transformers`/`torch` jamais installés dans `base`).

## 1. Charger le graphe (checkpoint natif NeuroDSL, déjà validé)

In [ ]:
using NeuroDSL, JSON

const MODEL_DIR = joinpath(@__DIR__, "qwen2.5-1.5b-instruct")
const DIM, N_LAYERS, N_HEADS, N_KV_HEADS, HIDDEN_DIM, VOCAB_SIZE = 1536, 28, 12, 2, 8960, 151936
const ROPE_THETA, RMS_EPS = 1_000_000.0, 1e-6

println("Construction du graphe (namespace UNIQUE -- sert les DEUX rôles : passage avant batché du préfixe ET décodage incrémental avec cache KV, voir le correctif 2026-08-31 ci-dessous)...")
dev = NeuroDSL.Backend.CUDADevice()
# CORRECTIF 2026-08-31 (voir notebook/diag_kv_cache_double_alloc_phaseA.jl / _phaseB.jl,
# mesuré sur ce modèle réel) : ns_load et ns_chat pointent maintenant vers LE
# MÊME namespace. Historiquement (2026-07-29) ils étaient séparés parce que
# `demand!` parcourait alors tout le PRÉFIXE TOPOLOGIQUE d'un namespace avant
# d'atteindre sa cible -- partager un namespace pouvait donc ré-exécuter un
# nœud `:kv_cache_append` à l'insu de l'appelant, avec un `cur_step` périmé,
# et corrompre `aux_data[:history]`. Le correctif de l'époque (garder deux
# namespaces + `copy_params_to_namespace!` pour dupliquer TOUS les poids par
# valeur d'un namespace à l'autre) double le budget mémoire des poids --
# mesuré : ~6.78 Go de poids supplémentaires, VRAM totale ~14.0-14.4 Go dès la
# construction du graphe de chat, AVANT même le premier tour de conversation.
# Depuis, `demand!` a été corrigé pour ne parcourir que le cône des ANCÊTRES
# RÉELS de sa cible (`_ancestors_of!`, src/graph_api.jl -- correctif écrit
# explicitement pour éliminer cette classe de bug), ce qui rend le partage
# d'un namespace unique SÛR : un `demand!` sur `logits_load` (racine
# `:token_ids`) ne touche jamais les nœuds `:kv_cache_append` du sous-graphe
# de décodage (racine `:dec_token_id`), et réciproquement -- aucun chemin
# topologique ne les relie hormis les nœuds de POIDS, qui n'ont pas de règle
# à ré-exécuter. Vérifié : les 34 tokens générés et les logits à chaque pas
# sont NUMÉRIQUEMENT IDENTIQUES (écart abs max = 0.0) entre l'ancien chemin à
# deux namespaces et celui-ci, sur le même prompt, poids réels Qwen2.5-1.5B --
# et le pic VRAM mesuré tombe de 14420 Mo à 10324 Mo (-4.1 Go, -28.4%), sans
# régression de vitesse (0.475 vs 0.488 s/tok en régime établi, dans le bruit).
const ns_load = :qwen2   # namespace unique : passage batché du préfixe ET cache KV
const ns_chat = ns_load  # alias -- gardé pour ne pas renommer tout le reste du notebook
g = NeuroDSL.NeuroGraph(namespace=ns_load, device=dev)
NeuroDSL.set!(g, :token_ids, ones(Int, 8); atom_type=NeuroDSL.Datom, namespace=ns_load)
tok_emb = NeuroDSL.Embedding(VOCAB_SIZE, DIM)(g, :token_ids, :tok; namespace=ns_load)
out_sym = NeuroDSL.LlamaModel(N_LAYERS, DIM, N_HEADS, HIDDEN_DIM;
                               batched_attn=true, n_kv_heads=N_KV_HEADS,
                               qkv_bias=true, use_rope=true, rope_theta=ROPE_THETA)(g, tok_emb; namespace=ns_load)
final_norm = NeuroDSL.LayerNorm(DIM; eps=RMS_EPS)(g, out_sym, :final_norm; namespace=ns_load)
const logits_load = NeuroDSL.Linear(DIM, VOCAB_SIZE, bias=false)(g, final_norm, :lm_head; namespace=ns_load)
# `logits_load` capturé (contrairement à la version précédente) : cell 3 en a
# besoin pour faire le passage avant BATCHÉ du préfixe -- voir plus bas.

println("Chargement des poids (qwen2_neurodsl.json/.bin, format natif -- pas de re-parsing du .safetensors)...")
NeuroDSL.load_graph!(g, ns_load, joinpath(MODEL_DIR, "qwen2_neurodsl"); overwrite=true)

# CORRECTIF MÉMOIRE 2026-08-31 (revue algorithmique complète du cache KV,
# voir notebook/diag_tied_embedding_alias.jl / _before_run.log / _after_run.log,
# notebook/diag_kv_cache_final_correctness_vram.jl / _run.log / _run2.log) :
# tok_E et lm_head_W sont des poids LIÉS (tie_word_embeddings=true, config.json
# Qwen2.5) -- load_qwen2.jl les pointe déjà vers le MÊME objet Julia à la
# conversion, mais `save_graph!`/`load_graph!` (src/serialization.jl) n'ont
# aucune notion d'aliasing entre deux symboles différents : le .bin contient
# deux blobs séparés de ~890 Mio identiques (vérifié : qwen2_neurodsl.json,
# offsets différents pour tok_E et lm_head_W), et `load_graph!` recrée donc
# deux CuArray indépendants -- l'alias d'origine est perdu au chargement, pas
# à la construction. `alias_tied_param!` (src/graph_api.jl, nouveau) vérifie
# l'égalité numérique (`isapprox`) puis réassigne lm_head_W.value au MÊME
# objet que tok_E, libère l'ancien buffer (`Backend.free!`) -- mesuré :
# pool_current -890 Mio (6779->5889 Mo), logits identiques (écart abs max =
# 0.0) avant/après sur le même prompt. Combiné à deux autres correctifs
# mesurés cette même passe (`load_graph!` libère désormais explicitement les
# anciens buffers avant d'en recréer, et `LlamaModel` nettoie périodiquement
# pendant la construction à poids aléatoires -- src/serialization.jl et
# src/layers.jl) : pic VRAM mesuré bout-en-bout sur une génération complète
# 10324 Mo -> 8667-8671 Mo (-1.66 Go, -16.1% de plus que le correctif du
# namespace unique déjà en place ; -5.75 Go / -39.9% depuis le tout premier
# état à deux namespaces + poids liés dupliqués). Génération/logits vérifiés
# identiques (écart abs max = 0.0) sur le même prompt à chaque étape.
NeuroDSL.alias_tied_param!(g, ns_load, :tok_E, :lm_head_W)

# PAS de copy_params_to_namespace! -- `build_cached_decode_graph!`/
# `CachedLlamaModel` (src/layers.jl) retrouvent par CONVENTION DE NOMMAGE les
# nœuds de poids déjà créés par `LlamaModel` ci-dessus, DANS LE MÊME
# namespace : même objet Julia, zéro octet supplémentaire, voir le correctif
# détaillé plus haut.
println("Construction du graphe de décodage incrémental avec cache KV (même namespace :$ns_load, poids PARTAGÉS, zéro copie)...")
const dec_logits = NeuroDSL.build_cached_decode_graph!(g;
    n_layers=N_LAYERS, dim=DIM, n_heads=N_HEADS, hidden_dim=HIDDEN_DIM, vocab_size=VOCAB_SIZE,
    n_kv_heads=N_KV_HEADS, qkv_bias=true, use_rope=true, rope_theta=ROPE_THETA, namespace=ns_chat)

# CORRECTIF PERF 2026-08-31 (ter) -- warm-up JIT du graphe de décodage caché,
# déplacé ICI (au chargement) au lieu d'être payé sur le premier message réel
# de l'utilisateur. Creusé à la demande explicite du coordinateur sur DEUX
# pistes (voir notebook/diag_decode_op_breakdown.jl, diag_embedding_gpu_gather_check.jl,
# diag_gpu_clock_correlate.jl, diag_warmup_jit_vs_clock.jl) :
#
# 1. Piste "attention batchée absente du décodage" : confirmée factuellement
#    (`CachedMultiHeadAttention`, src/layers.jl, boucle un `:matmul` par tête,
#    jamais `:batched_qk`/`:batched_pv`) mais mesurée SECONDAIRE une fois le
#    vrai goulot (point 3 ci-dessous) corrigé : microbenchmark CUDA brut
#    (`diag_decode_batched_attn_microbench.jl`) donne ~10ms/token de gain
#    potentiel (~1.45x sur QK+PV seuls), MAIS l'attention de décodage n'a PAS
#    la même forme que celle du training (1 requête contre `cur_step` clés
#    en cache, pas seq=seq) -- `batched_qk_fwd!`/`batched_pv_fwd!` existants
#    (src/kernels.jl) supposent Q et K de même longueur et ne s'appliquent
#    PAS tels quels ; les buffers K/V du cache sont réalloués frais à chaque
#    pas (voir kv_cache.jl), donc le chemin batché retomberait TOUJOURS sur
#    un `_gather3` (copie), jamais le zero-copy utilisé en training. Un
#    nouveau noyau à forme asymétrique serait nécessaire pour un gain de
#    ~4-5% du temps par token une fois le correctif ci-dessous appliqué --
#    net à ce stade, mais pas implémenté cette session (bar du projet :
#    corriger ce qui compte vraiment, pas empiler des gains marginaux
#    risqués) ; laissé comme piste documentée pour une session future.
#
# 2. Piste "horloge GPU" : RÉFUTÉE comme cause de ralentissement pendant la
#    génération -- `diag_gpu_clock_correlate.jl` corrèle la durée RÉELLE de
#    chaque pas avec l'horloge SM mesurée par `nvidia-smi` pendant une vraie
#    génération : des pas à 210MHz et des pas à 1500MHz prennent le MÊME
#    temps (~420-480ms) -- le décodage n'est PAS limité par l'horloge/le
#    calcul GPU, confirmant et RENFORÇANT (avec des mesures directes, pas
#    une supposition) la conclusion "kernel-launch-count driven" d'une passe
#    de profilage antérieure.
#
# 3. LE VRAI GOULOT (trouvé en creusant plus loin, pas anticipé) : l'op
#    `:embedding` (src/dispatch.jl, AVANT ce correctif) faisait
#    `E_cpu = Array(E)` -- un aller-retour GPU->CPU de la table d'embedding
#    ENTIÈRE (890 Mo, vocab=151936 x dim=1536) À CHAQUE lookup, y compris
#    pour UN SEUL token (`:dec_tok_emb`, appelé une fois PAR TOKEN GÉNÉRÉ).
#    Mesuré : ~220ms par appel (`diag_embedding_gpu_gather_check.jl`) --
#    c'est CE transfert, pas l'attention, qui dominait le profil précédent
#    ("attention ≈8x MLP") : l'embedding du nouveau token est un ANCÊTRE du
#    premier checkpoint de la couche 1, donc son coût lui était attribué à
#    tort. Corrigé dans `src/dispatch.jl` (`view(E, idx_cpu, :)`, gather GPU
#    natif au lieu du memcpy complet) : 0.021ms au lieu de 220ms (~10 600x
#    sur cet op seul), résultat bit-identique (`isapprox` vérifié). Mesuré
#    bout-en-bout sur ce même notebook : ~0.44-0.50s/tok -> ~0.22-0.26s/tok
#    en régime établi (~2x). Correction vérifiée en DEUX process Julia
#    isolés (avant/après, `git stash` du seul correctif) : mêmes 40 tokens
#    générés, même texte, logits bit-identiques (écart abs max = 0.0) --
#    voir diag_embedding_fix_correctness_{before,after}_results.json.
#
# CE WARM-UP (pas de conséquence sur le cache réel : `prime_kv_cache_from_prefix!`,
# plus bas, écrase entièrement `aux_data[:history]` de chaque nœud de cache
# avec le VRAI préfixe avant toute génération -- vérifié dans
# diag_warmup_jit_vs_clock.jl, mêmes tokens/logits avec et sans ce warm-up)
# absorbe la compilation JIT de première invocation des ops personnalisés du
# décodage caché (`:kv_cache_append`/`:scale_no_mask`/`:rope_at_pos`, jamais
# exercés par le passage avant batché du préfixe ci-dessus, qui utilise un
# graphe différent) PENDANT le chargement plutôt que pendant la première
# réponse. Mesuré : premier token réel 2163ms (sans ce warm-up) -> 492ms
# (avec, régime établi ~220-260ms) -- le warm-up lui-même coûte ~8.5s, payé
# ici une seule fois, pas par l'utilisateur.
println("Warm-up du graphe de décodage (1 pas jetable -- absorbe la compilation JIT de première invocation pendant le chargement, pas pendant la première réponse)...")
const _t_warmup = @elapsed begin
    NeuroDSL.set!(g, :dec_token_id, [1]; atom_type=NeuroDSL.Datom, namespace=ns_chat)
    NeuroDSL.set!(g, :dec_cur_step, Float32[1]; namespace=ns_chat)
    NeuroDSL.set!(g, :dec_pos, Float32[0]; namespace=ns_chat)
    NeuroDSL.invalidate_all!(g; namespace=ns_chat)
    Array(NeuroDSL.demand!(g, dec_logits; namespace=ns_chat))
end
println("Warm-up terminé (", round(_t_warmup, digits=1), "s) -- prêt pour le premier vrai prompt.")

println("Prêt -- poids partagés par référence (aucune copie). Un seul namespace : passage batché du préfixe ET cache KV pour la génération.")

## 2. Pont tokenizer (process externe, environnement conda isolé, zéro nouvelle dépendance Julia)

In [2]:
const PYTHON_ENV = raw"C:\Users\Nevermind\anaconda3\envs\neurodsl_llm_check\python.exe"
const TOKENIZER_HELPER = joinpath(@__DIR__, "qwen2_tokenizer_helper.py")

# Serveur persistant (correctif du 2026-07-29, voir cell 0) : `transformers` +
# le tokenizer ne sont chargés QU'UNE FOIS, au démarrage de ce process, qui
# reste ensuite vivant pour toute la durée de la session -- au lieu de
# relancer un interpréteur Python + réimporter `transformers` + recharger le
# tokenizer depuis le disque à CHAQUE appel (`encode_chat`, `decode_ids`),
# comme le faisait la version précédente (~13-14s de coût fixe PAR appel,
# mesuré ce soir, indépendant du cache KV -- voir kv_cache_chat_timing_probe_run2.log
# et kv_cache_tokenizer_persistent_check_run.log pour la vérification de
# correction : mêmes résultats byte pour byte que le mode un-coup historique
# sur eos_id/encode_chat/decode, sur plusieurs historiques réels).
println("Démarrage du serveur tokenizer persistant (transformers + tokenizer chargés UNE fois)...")
const TOKENIZER_PROC = open(`$PYTHON_ENV -u $TOKENIZER_HELPER --serve`, "r+")
const _tok_ready = JSON.parse(readline(TOKENIZER_PROC))
println("Serveur tokenizer prêt : ", _tok_ready)

function _call_helper(req::Dict)
    println(TOKENIZER_PROC, JSON.json(req))
    flush(TOKENIZER_PROC)
    return JSON.parse(readline(TOKENIZER_PROC))
end

"""Encode l'historique de messages via le VRAI gabarit ChatML du tokenizer
(`apply_chat_template`, add_generation_prompt=true) -- pas une réimplémentation
manuelle de la chaîne `<|im_start|>...`."""
encode_chat(messages) = Int.(_call_helper(Dict("action"=>"encode_chat", "messages"=>messages))["ids"])

"""Décode une suite d'IDs (0-indexés, convention HuggingFace) en texte lisible,
en omettant les tokens spéciaux (`<|im_end|>` etc.)."""
decode_ids(ids::Vector{Int}) = _call_helper(Dict("action"=>"decode", "ids"=>ids))["text"]

const EOS_ID = _call_helper(Dict("action"=>"eos_id"))["id"]
println("Token de fin de tour <|im_end|> = ", EOS_ID, " -- confirmé depuis le tokenizer réel, pas codé en dur à l'aveugle.")

Démarrage du serveur tokenizer persistant (transformers + tokenizer chargés UNE fois)...
Serveur tokenizer prêt : Dict{String, Any}("ready" => true)
Token de fin de tour <|im_end|> = 151645 -- confirmé depuis le tokenizer réel, pas codé en dur à l'aveugle.


## 3. Génération avec cache KV (préfixe batché + décodage incrémental) + `chat(prompt)` multi-tours

Deux phases, PAS une seule boucle uniforme (correctif du 2026-07-29 -- voir cell 0) :
1. **Préfixe** (historique ChatML ré-encodé, plusieurs dizaines de tokens) : UN SEUL passage avant batché sur `ns_load` (recalcul complet, exactement comme n'importe quel forward pass), qui amorce directement le cache KV de `ns_chat` via `NeuroDSL.prime_kv_cache_from_prefix!` -- pas de boucle token par token pour cette partie.
2. **Génération** (les nouveaux tokens de la réponse, un par un -- chacun dépend forcément du précédent) : `set!` + `invalidate_all!` + `demand!` par token sur `ns_chat`, le cache porte l'historique.

La frontière préfixe→génération est invisible pour le cache lui-même (le mécanisme d'amorçage produit EXACTEMENT le même état de cache qu'un remplissage token par token -- vérifié par `notebook/kv_cache_prefix_prime_qwen_gate.jl`, écart max `5.6e-5`, continuation identique sur 5 pas). La boucle de génération s'arrête toujours sur le VRAI token `<|im_end|>` (`EOS_ID`) plutôt qu'après un nombre fixe de tokens.

In [3]:
"""Avance le cache KV d'UN token (`tok0`, 0-indexé HuggingFace) à la position
`cur_step` (1-indexée) et retourne les logits `(vocab,)` qui en résultent --
la prédiction du token SUIVANT `tok0`. Réservé aux tokens GÉNÉRÉS un par un
(chacun dépend du précédent) -- PAS au préfixe, qui est traité en un seul
passage batché ci-dessous (voir cell 0 -- correctif de performance du
2026-07-29)."""
function cached_step!(g, tok0::Int, cur_step::Int)
    NeuroDSL.set!(g, :dec_token_id, [tok0 + 1]; atom_type=NeuroDSL.Datom, namespace=ns_chat)
    NeuroDSL.set!(g, :dec_cur_step, Float32[cur_step]; namespace=ns_chat)
    NeuroDSL.set!(g, :dec_pos, Float32[cur_step-1]; namespace=ns_chat)
    NeuroDSL.invalidate_all!(g; namespace=ns_chat)
    return Array(NeuroDSL.demand!(g, dec_logits; namespace=ns_chat))[1, :]
end

const HISTORY = Dict{String,Any}[]  # conversation multi-tours : [{"role"=>..,"content"=>..}, ...]

"""
    chat(prompt::String; max_new_tokens=100, verbose=true) -> String

Ajoute `prompt` comme tour utilisateur à `HISTORY`, encode TOUT l'historique
via le gabarit ChatML réel, traite ce préfixe en UN SEUL passage avant batché
(`ns_load`, recalcul complet -- comme n'importe quel forward pass), amorce le
cache KV (`ns_chat`) avec le résultat via `NeuroDSL.prime_kv_cache_from_prefix!`,
puis génère la réponse de l'assistant token par token via le cache (gloutonne,
arrêt sur `<|im_end|>` ou `max_new_tokens`), l'ajoute à `HISTORY` à son tour,
et retourne le texte décodé.
"""
function chat(prompt::AbstractString; max_new_tokens::Int=100, verbose::Bool=true)
    push!(HISTORY, Dict("role"=>"user", "content"=>String(prompt)))
    ids0 = encode_chat(HISTORY)   # 0-indexé (HuggingFace)
    prefix = ids0 .+ 1             # 1-indexé pour NeuroDSL::Embedding

    gen0 = Int[]
    stopped_on_eos = false
    t0 = time()

    # -- Préfixe : UN passage avant batché (ns_load), PAS une boucle token
    # par token -- c'est exactement ce que corrige `prime_kv_cache_from_prefix!`
    # par rapport à la première version de ce notebook (~0.5s PAR TOKEN du
    # préfixe -> ~0.5s pour TOUT le préfixe, quelle que soit sa longueur).
    NeuroDSL.set!(g, :token_ids, prefix; atom_type=NeuroDSL.Datom, namespace=ns_load)
    NeuroDSL.invalidate_all!(g; namespace=ns_load)
    prefix_out = Array(NeuroDSL.demand!(g, logits_load; namespace=ns_load))
    logits_row = Float32.(prefix_out[end, :])   # prédiction du 1er token de la réponse
    NeuroDSL.prime_kv_cache_from_prefix!(g; src_ns=ns_load, dst_ns=ns_chat,
        n_layers=N_LAYERS, n_kv_heads=N_KV_HEADS, use_rope=true)
    cur_step = length(prefix)

    # -- Génération, un token à la fois via le cache incrémental --
    for step in 1:max_new_tokens
        nxt0 = argmax(logits_row) - 1
        if nxt0 == EOS_ID
            stopped_on_eos = true
            break
        end
        push!(gen0, nxt0)
        cur_step += 1
        logits_row = cached_step!(g, nxt0, cur_step)
    end
    dt = time() - t0
    reply = decode_ids(gen0)
    push!(HISTORY, Dict("role"=>"assistant", "content"=>reply))
    if verbose
        println("  [", length(gen0), " tokens, ", round(dt, digits=1), "s",
                 stopped_on_eos ? ", arrêt sur <|im_end|>" : ", tronqué à max_new_tokens", "]")
    end
    return reply
end

"""Vide l'historique de conversation -- à appeler pour repartir d'un échange neuf."""
reset_chat!() = (empty!(HISTORY); println("Historique vidé."))

println("chat(...) prêt -- préfixe en un passage batché, génération via cache KV.")

chat(...) prêt -- préfixe en un passage batché, génération via cache KV.


## 4. Essayer

Chaque cellule ci-dessous peut être ré-exécutée avec un nouveau texte. `reset_chat!()` avant de changer de sujet si vous ne voulez pas que l'ancien échange reste dans le contexte.

In [ ]:
reset_chat!()
t_cell8 = @elapsed rep_cell8 = chat("What is the capital of Egypt?")
println(rep_cell8)

In [ ]:
# Tour suivant DANS LA MÊME conversation -- teste si l'historique est vraiment réutilisé
println(chat("And what is a famous food from that city?"))

In [ ]:
# Nouvelle conversation, sujet différent
reset_chat!()
println(chat("Write a haiku about the ocean."))

In [ ]:
# Nouvelle conversation, sujet différent
reset_chat!()
println(chat("the minimum of  2,3,15,0"))

## 5. Comparaison contrôlée froid vs chaud, DANS CE NOTEBOOK (pas un script séparé)

Le coordinateur a signalé un désaccord réel : l'utilisateur a appelé `chat(...)` deux fois dans le même noyau et n'a observé "aucune vraie différence", alors que `kv_cache_chat_timing_probe_warm.jl` (un script séparé) montrait un facteur ~15x entre le premier et le deuxième appel. Les cellules 8/9 ci-dessus ne sont PAS une comparaison équitable : 7 tokens générés contre 37 -- le total brut (14.6s vs 21.7s) donne l'impression fausse d'un ralentissement, alors que normalisé par token c'est déjà ~3.8x plus rapide (2.09s/tok -> 0.59s/tok). Les deux cellules ci-dessous répètent EXACTEMENT le même prompt (même nombre de tokens de préfixe, réponse attendue identique en décodage glouton) pour une comparaison vraiment équitable, directement dans ce notebook, pas dans un script à part.

In [ ]:
# Appel A -- même prompt que la cellule 8, mais réexécuté ICI, plus tard dans
# la même session (kernel jamais redémarré depuis le début du notebook) --
# donc DÉJÀ chaud au sens du JIT (cellule 8 a déjà exercé ce chemin de calcul).
reset_chat!()
t_a = @elapsed rep_a = chat("What is the capital of Egypt?")
println("Appel A : ", round(t_a, digits=1), "s -- ", rep_a)

In [ ]:
# Appel B -- répète EXACTEMENT le même prompt une seconde fois (warm vs warm,
# stabilité) -- si l'appel A ci-dessus est déjà chaud, B doit être du même
# ordre de grandeur que A, pas plus lent.
reset_chat!()
t_b = @elapsed rep_b = chat("What is the capital of Egypt?")
println("Appel B : ", round(t_b, digits=1), "s -- ", rep_b)
println("\nComparaison directe (même prompt \"What is the capital of Egypt?\", même réponse attendue), temps TOTAL (tokenizer inclus) :")
println("  Cellule 8 (1er appel EVER de la session -- cache/JIT froid ET tokenizer froid) : ", round(t_cell8,digits=1), "s")
println("  Appel A ci-dessus (session déjà chaude depuis la cellule 8)                     : ", round(t_a,digits=1), "s")
println("  Appel B ci-dessus (encore un appel plus tard, même session)                     : ", round(t_b,digits=1), "s")

In [6]:
using CUDA
GC.gc()       # Julia repère les tenseurs morts et les met à la poubelle
GC.gc()       # (Un 2ème passage est souvent utile pour les objets complexes)
CUDA.reclaim() # CUDA vide enfin la poubelle de la carte graphique

In [ ]:
# ==========================================
# CELLULE D'INTERFACE CHAT INTERACTIVE
# ==========================================
println("--- Démarrage du Chat Interactif avec Qwen2.5-1.5B (NeuroDSL) ---")
println("-> Tapez 'exit' ou 'quit' pour arrêter.")
println("-> Tapez 'reset' pour vider l'historique (repartir à zéro).")

# Amorçage à froid (pour payer le coût JIT de 14.2s une seule fois au lancement)
println("\n[Amorçage du moteur et du JIT CUDA en cours...]")
chat("Hello"; max_new_tokens=1, verbose=false)
reset_chat!()
println("[Moteur chaud et prêt. Temps de réponse attendu : ~4.1s]")

while true
    print("\n🧑 Vous : ")
    user_input = readline()
    
    if lowercase(strip(user_input)) in ["exit", "quit"]
        println("Fin de la session.")
        break
    elseif lowercase(strip(user_input)) == "reset"
        reset_chat!()
        continue
    elseif isempty(strip(user_input))
        continue
    end
    
    print("🤖 Qwen : ")
    # Appel de la fonction chat optimisée avec le cache KV -- IMPORTANT : chat()
    # RETOURNE le texte de la réponse (`return reply`), il ne l'affiche jamais
    # lui-même (verbose=false désactive même le résumé de timing). Sans
    # capturer et imprimer cette valeur de retour, la génération se déroule
    # bien en interne mais rien n'apparaît à l'écran -- c'est exactement le
    # bug qui donnait l'impression que le modèle "ne répondait pas".
    reply = chat(user_input; max_new_tokens=200, verbose=false)
    println(reply)
    println() # Saut de ligne pour la lisibilité
end

--- Démarrage du Chat Interactif avec Qwen2.5-1.5B (NeuroDSL) ---
-> Tapez 'exit' ou 'quit' pour arrêter.
-> Tapez 'reset' pour vider l'historique (repartir à zéro).

[Amorçage du moteur et du JIT CUDA en cours...]
Historique vidé.
[Moteur chaud et prêt. Temps de réponse attendu : ~4.1s]

🧑 Vous : 

stdin>  hi 


🤖 Qwen : Hello! How can I assist you today?


🧑 Vous : 

stdin>  where are you from ?


🤖 Qwen : I'm an artificial intelligence designed to assist with a variety of tasks and answer questions. I don't have a physical location or a specific origin. I'm located in the cloud and can be accessed from anywhere in the world. Is there something specific you'd like to know or discuss?


🧑 Vous : 

...

🤖 Qwen : The result of 5 multiplied by 6 is 30.


🧑 Vous : 

stdin>  gcd of  120 and 180 


🤖 Qwen : The greatest common divisor (GCD) of 120 and 180 is 60.


🧑 Vous : 

stdin>  give me the step by step for counting the gcd of 120 and 144


🤖 Qwen : Certainly! To find the greatest common divisor (GCD) of 120 and 144, we can use the Euclidean algorithm, which involves repeated division. Here are the steps:

1. **Divide the larger number by the smaller number and find the remainder:**
   - Divide 144 by 120.
   - \( 144 \div 120 = 1 \) with a remainder of \( 144 - 120 = 24 \).

2. **Replace the larger number with the smaller number and the smaller number with the remainder from the previous step:**
   - Now, we need to find the GCD of 120 and 24.

3. **Repeat the process:**
   - Divide 120 by 24.
   - \( 120 \div 24 = 5 \) with a remainder of \( 12


🧑 Vous : 